# **Filtrando sequências para análises**

Filtrando sequências de genomas de vírus da raiva por comprimento da sequência e presença de rótulos de localidade e hospedeiro.

Foi realizado o download de arquivo Genbank (.gb) de todos os genomas depositados para filtrar sequências por tamanho e por seus metadados.

In [ ]:
# instalar BioPython
# pip install biopython 

In [ ]:
# importar pacotes e funcoes necessarias
from Bio import SeqIO
from function_seq_filter_list import seq_filter_list

In [ ]:
# caminho para arquivo com metadados das sequencias
path_seqs = r"lyssavirus.gb"

In [ ]:
# numero de sequencias no arquivo 
records = list(SeqIO.parse(path_seqs, "genbank"))
len(records)

In [ ]:
# filtrando sequencias de comprimento 11000 to 12000
seqs_length = []
for record in SeqIO.parse(path_seqs, "genbank"):
        length = len(record.seq) 
        if length > 11000 and length < 12000:
                seqs_length.append(record)
# numero de sequencias apos filtro
len(seqs_length)

In [ ]:
# selecionando sequencias que possuem rotulo de hospedeiro
seqs_host = seq_filter_list(seqs_length, "host")
len(seqs_host)

In [ ]:
# selecionando sequencias que possuem rotulo de localidade
seqs_locality = seq_filter_list(seqs_host, "geo_loc_name")
len(seqs_locality)

In [ ]:
# selecionando sequencia que possuem rotulo de data de coleta
seqs_date = seq_filter_list(seqs_locality, "collection_date")
len(seqs_date)

In [ ]:
# salvar arquivo em formato de metadados GenBank e formato FASTA
SeqIO.write(seqs_date,"lyssavirus_filtrado.gb","genbank") # 3189
SeqIO.write(seqs_date,"lyssavirus_filtrado.fasta","fasta") 

In [ ]:
# obtendo informações das sequências filtradas e convertando-as para uma tabela
# sequências duplicadas mantidas

# criar lista que vai receber dataframe   
data = []
# iterar sobre arquivo GenBank filtrado
for record in SeqIO.parse("lyssavirus_filtrado.gb", "genbank"):
    genome_info = {
        "accession": record.id,
        "sequence": record.seq,
        "length": len(record.seq),
        "organism": "",
        "geographical_location": "",
        "host": "",
        "authors": "",
        "title": "",
        "journal": ""
    }
    
    # extrair organismo, data de coleata, localidade geográfica e hospedeiro
    for feature in record.features:
        if feature.type == "source":
            genome_info["organism"] = feature.qualifiers.get('organism', [''])[0]
            genome_info["collection_date"] = feature.qualifiers.get('collection_date', [''])[0]
            genome_info["geographical_location"] = feature.qualifiers.get('geo_loc_name', [''])[0]
            genome_info["host"] = feature.qualifiers.get('host', [''])[0]
    
    # extrair autores e título do artigo das referências
    if record.annotations.get('references'):
        for ref in record.annotations['references']:
            if hasattr(ref, 'authors'):
                genome_info["authors"] = ', '.join(ref.authors)
            if hasattr(ref, 'title'):
                genome_info["title"] = ref.title
            if hasattr(ref, 'journal'):
                genome_info["journal"] = ref.journal
    
    data.append(genome_info)

In [ ]:
# criar dataframe com as informações recuperadas
df = pd.DataFrame(data)
df.shape

In [ ]:
# checar dataframe final
df.head()

In [ ]:
# padronizar formato de data de coleta e recuperando anos das datas
df['date'] = pd.to_datetime(df['collection_date'], format='mixed', errors='coerce')
df['year'] = df['date'].dt.strftime('%Y')

In [ ]:
# salvar em arquivo CSV
df.to_csv("rabies_final_20250916.csv", index=False)